In [ ]:
%load_ext autoreload
%autoreload 2

<a name="top"></a>
# Ice Sheet Simulation Compliance Checker

## Abstract

This tool checks the compliance of ISMIP6 netCDF formatted simulation experiment files according criteria, which are related to:<br />

- naming conventions
- admissible numerical values
- spatial definition of the grid which differs according to the ice sheet, Greenland (GIS) vs Antarctica (AIS)
- time recording dependent of the experiments

This tool was ported from the GitHub [Ice Sheet Simulation compliance checker](https://github.com/jbbarre/ISM_SimulationChecker) project, initially developed by Jean-Baptiste Barré as part of the PROTECT project.<br />
The PROTECT project has received funding from the European Union’s Horizon 2020 research and innovation programme under grant agreement No 869304.<br />

## Overview

- The compliance criteria of output variables and the compliance criteria of experiments are defined in separate csv files. 
- The variable compliance criteria are following the conventions defined in the [ISMIP6 Wiki](https://theghub.org/groups/ismip6/wiki/MainPage). 
- You select the experiment compliance criteria from `ISMIP6-Projections-Greenland`, `ISMIP6-Projections-Antarctica` and `ISMIP6-Projections2300-Antarctica`<br />
Tier 1 experimental setups defined in the [ISMIP6 Wiki](https://theghub.org/groups/ismip6/wiki/MainPage). See [ISMIP6_EXP_ALL](https://docs.google.com/spreadsheets/d/1YEiPV3Uc0K8EqD57mz-ZqBIF4XnVb5UbyRDRmox0KeE/edit#gid=1531275537) and [ISMIP6_EXP_AIS_2300](https://docs.google.com/spreadsheets/d/1J1AjPrnOLqXu-n8BJeYjZqdzXKv6g3wGRWmZt0tGuNc/edit#gid=0) for more information.
- You upload a ISMIP6 netCDF formatted simulation experiment file from your personal computer and the file is checked for compliance automatically.

Please [contact us](https://theghub.org/support/ticket/new) if you need help with running this tool.<br />


## Tool Output

- A compliance report will populate this notebook once the compliance check completes.

In [ ]:
# As of 12/2023, tested with the Jupyter Notebooks (202210) Tool and the geospatial-plus python3 Kernel
# arbitrary comment 

import sys,os,json,glob
import math
import getpass
import platform
import subprocess
import shutil
import os
from os import listdir
from os.path import isfile, join
import ipywidgets as widgets
#import hublib.ui as ui
import time
from datetime import datetime
import pandas as pd
import atexit
import signal
import numpy as np
from tabulate import tabulate
from ipywidgets import FileUpload

from ipywidgets import HBox

from IPython.display import HTML, display, Javascript

#import hublib.use

# Set up the environment for this notebook

# Setup paths to executables
scriptpath = os.path.realpath(" ")
        
# Get the parent dirs
self_tooldir = os.path.dirname(scriptpath)

# Setup path to python and bash scripts
self_bindir = os.path.join(self_tooldir, "bin")

# Add to PYTHONPATH
sys.path.insert (1, self_bindir)

# Setup path to python and bash scripts
self_remotebindir = os.path.join(self_tooldir, "remotebin")

# Set up path to the current data directory
self_datadir = os.path.join(self_tooldir, "data")

# Set up path to the current doc directory
self_docdir = os.path.join(self_tooldir, "doc")

# Set up path to the current session directory
self_workingdir = os.getcwd()

# Set up path to the user's home directory
self_homedir = os.path.expanduser("~")

# Initialize the dated run directory.
# Workflow results are not available until after a workflow is executed via Pegasus and completes
self_rundir = ""

self_user = getpass.getuser()

np.set_printoptions(threshold=np.inf) 

from compliance_checker_ghub import compliance_checker_ghub

# Downloaded hublib.ui.upload.py from https://pypi.org/project/hublib/0.9.97/#files.
# Modified upload.py's  _filenames_received function to check the received filename, consistent with compliance_checker_ghub checks, before continuing the upload.
# Modified upload.py to add a _filenames_received user callback function.
# Modified upload.py to trigger change event when the same file is uploaded again, the received filename check depends on the currently selected experiment criteria.
upload = None
from upload import FileUpload
from upload import upload_status_check_success
from upload import upload_status_ext_check_failed
from upload import upload_status_exp_check_failed
from upload import upload_status_var_check_failed
# Downloaded hublib.ui.download.py from https://pypi.org/project/hublib/0.9.97/#files.
from download import Download

self_log_filepath = os.path.join(self_workingdir, 'isschecker_log_file.txt')
self_log_snapshot_filepath = os.path.join(self_workingdir, 'isschecker_log_snapshot_file.txt')
self_log_backup_filepath = os.path.join(self_workingdir, 'isschecker_log_backup_file.txt')

widget_border_style = '1px solid black'
widget_output_border_style = '1px solid black'

BOLD = '\033[1m'
SUCCESS = '\033[92m'
WARNING = '\033[93m'
FAIL = '\033[91m'
END = '\033[0m'

dropdown_width = '965px'
dropdown_height = '30px'
button_width = '250px'
button_height = '35px'
button_width2 = '350px'
button_height2 = '35px'


<a name="user_guide"></a>
## User Guide

### [**Steps for using this tool**](#steps_for_using_this_tool)<br />

1. [View the variable compliance criteria](#view_variable_criteria)
2. [Select the experiment compliance criteria](#select_experiment_criteria)
3. [View the experiment compliance criteria](#view_experiment_criteria)
4. [Upload your netCDF formatted simulation experiment file](#upload_file) 
5. [View and download compliance checker output](#view_output)
6. [View the log output file](#view_log_output)

### [**Background**](#background)<br />

In [ ]:
main_display = widgets.Output()
display (main_display)

In [ ]:
#hideCodeButton = hublib.ui.HideCodeButton(style='success')
#display(hideCodeButton)

In [ ]:
experimental_setup_list = ['ISMIP6-Projections-Greenland', 'ISMIP6-Projections-Antarctica','ISMIP6-Projections2300-Antarctica']
experimental_setup_csv_filenames = ['ISMIP6-Projections-Greenland.csv','ISMIP6-Projections-Antarctica.csv','ISMIP6-Projections2300-Antarctica.csv']
experimental_setup_index = 1

In [ ]:
# Clean up.
# Per https://docs.python.org/3/library/atexit.html, atexit handlers aren't executed if the program is killed by a signal not handled by Python, 
# or in case of internal error, or if os._exit() is called. 
def exit_handler(signal, frame):
    
    if exp_directory and os.path.exists(exp_directory):
        shutil.rmtree(exp_directory)
        
    FH1.flush()
    FH1.close()

    
atexit.register(exit_handler, None, None)

# For Appmode need to handle signal.SIGTERM
signal.signal(signal.SIGTERM, exit_handler);


In [ ]:
# prevent In[] and Out[] from displaying on left
#HTML('''
#<style>.prompt{width: 0px; min-width: 0px; visibility: collapse}</style>
#''')

In [ ]:
# Reference for on("load"): https://api.jquery.com/ready/
HTML('''
<script>
    function scroll_to_top() {
        Jupyter.notebook.scroll_to_top();
    } 
    $( window ).on( "load", scroll_to_top() );
</script>
''')

In [ ]:
# Button styles
HTML('''
<style>.buttontextclass { color:black ; font-size:120%}</style>
''')

In [ ]:
#https://stackoverflow.com/questions/36757301/disable-ipython-notebook-autoscrolling

In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [ ]:
if os.path.exists(self_log_filepath):
    shutil.move (self_log_filepath, self_log_backup_filepath)
    
FH1 = open(self_log_filepath, 'w')

show_log_output_button = widgets.Button(description="Show Log Output", disabled=False,\
    layout=widgets.Layout(width=button_width, height=button_height),\
    style= {'button_color':'lightgreen','font_weight':'bold'})

# Utility Functions

def log_info (message):
    
    if show_log_output_button.description == 'Hide Log Output': 
        with log_output:
            print (message)    
    FH1.write('%s\n' %message)
    FH1.flush()

def log_status (output_widget, message):
    
    with output_widget:
        print (message)
    log_info (message)
    
def log_success (output_widget, message):
    
    with output_widget:
        print ('%s%s%s' %(SUCCESS,message,END))
    log_info (message)
    
def log_warning (output_widget, message):
    
    with output_widget:
        print ('%s%s%s' %(WARNING,message,END))
    log_info (message)
    
def log_error (output_widget, message):
    
    with output_widget:
        print ('%s%s%s' %(FAIL,message,END))
    log_info (message)
    


<a name="steps_for_using_this_tool"></a>
## Steps for Using This Tool<a name="view_variable_criteria"></a>


### View the Variable Compliance Criteria

- Click the `Show Variable Compliance Criteria` button to view the variable compliance criteria.

In [ ]:
# load csv :
filename = 'ismip6_criteria.csv'
self_var_criteria_csv = os.path.join(self_workingdir, 'data', filename)
self_var_criteria_tabulated_csv = '%s_tabulated.txt' %filename
csv  = pd.read_csv(self_var_criteria_csv, delimiter=';', header='infer', decimal=",")
#print ('type(csv): ', type(csv)) #<class 'pandas.core.frame.DataFrame'>
#print ('csv: ', csv)
csv.drop(['type', 'standard_name', 'alias_name'], inplace=True, axis=1)
#print ('csv: ', csv)

self_ismip6_mandatory_variables = csv['variable'][csv.mandatory==1].tolist()
#print (type(self_ismip6_mandatory_variables))
#print (self_ismip6_mandatory_variables)

f = open(self_var_criteria_tabulated_csv, 'w')

tabulated_csv = tabulate(csv, headers='keys', tablefmt='grid')
#print (tabulated_table)
f.write(tabulated_csv)
f.flush()
f.close()


In [ ]:
def show_variable_criteria_output():
    
    if show_var_criteria_output_button.description == 'Hide Variable Compliance Criteria':

        var_criteria_output.clear_output()        

        with var_criteria_output:
            if os.path.exists(self_var_criteria_tabulated_csv):
                print("\n%s: \n" %self_var_criteria_csv)
                f = open(self_var_criteria_tabulated_csv,'r')
                for line in f:
                    print(line.rstrip())
                f.close()
            else:
                log_error (log_output, '%s does not exist ' %self_var_criteria_tabulated_csv + '. Please contact us.')

def show_variable_criteria_output_on_click(change):
    
    if os.path.exists(self_var_criteria_tabulated_csv):
            
        if show_var_criteria_output_button.description == 'Show Variable Compliance Criteria':
            
            show_var_criteria_output_button.description = 'Hide Variable Compliance Criteria'
            show_variable_criteria_output()
            
        else:
        
            show_var_criteria_output_button.description = 'Show Variable Compliance Criteria'
            var_criteria_output.clear_output()
    else:
        log_error (log_output, '%s does not exist ' %self_var_criteria_tabulated_csv + '. Please contact us.')

show_var_criteria_output_button = widgets.Button(description="Show Variable Compliance Criteria", disabled=False,\
    layout=widgets.Layout(width=button_width2, height=button_height2),\
    style= {'button_color':'lightgreen','font_weight':'bold'})

show_var_criteria_output_button.add_class("buttontextclass")
show_var_criteria_output_button.on_click(show_variable_criteria_output_on_click)
display (show_var_criteria_output_button)

In [ ]:
var_criteria_output = widgets.Output(layout={'border': widget_output_border_style})
display (var_criteria_output)

<a name="select_experiment_criteria"></a>
### Select the Experiment Compliance Criteria

- Select the experiment compliance criteria using the `Experimental Setup` dropdown.


In [ ]:
# load csv :
def load_experiment_setup_csv():
    
    global self_experimental_criteria_filename
    global self_exp_criteria_csv
    global self_exp_criteria_tabulated_csv
    global self_ismip6_experiments
    global upload

    self_experimental_criteria_filename = experimental_setup_csv_filenames[experimental_setup_index]
    self_exp_criteria_csv = os.path.join(self_workingdir, 'data', self_experimental_criteria_filename)
    self_exp_criteria_tabulated_csv = '%s_tabulated.txt' %self_experimental_criteria_filename
    csv  = pd.read_csv(self_exp_criteria_csv, delimiter=',', header='infer', decimal=".")
    #print ('type(csv): ', type(csv))
    #print (csv)
    self_ismip6_experiments = csv['experiment'].tolist()
    #print (type(self_ismip6_experiments)) #<class 'list'>
    #print (self_ismip6_experiments)
    #log_info ('self_ismip6_experiments: ' + str(self_ismip6_experiments))
    
    if (upload):
        upload.set_experiments(self_ismip6_experiments)

    f = open(self_exp_criteria_tabulated_csv, 'w')

    tabulated_csv = tabulate(csv, headers='keys', tablefmt='grid')
    #print (tabulated_table)
    f.write(tabulated_csv)
    f.flush()
    f.close()
    
load_experiment_setup_csv()


In [ ]:
def experimental_setup_dropdown_callback(change):
    
    global experimental_setup
    global experimental_setup_index
   
    if change['type'] == 'change' and change['name'] == 'value' and change['new'] != ' ' \
        and experimental_setup_dropdown.value != None:
        
        experimental_setup = experimental_setup_dropdown.value
        log_info ('experimental setup: ' + experimental_setup)
        experimental_setup_index = experimental_setup_list.index(experimental_setup)
        log_info ('experimental setup index: ' + str(experimental_setup_index))
        
        # load csv :
        load_experiment_setup_csv()
        show_experiment_criteria_output()
        #initialize()
        
experimental_setup_dropdown = widgets.Dropdown(
    description = 'Experimental Setup:',
    disabled = False,
    options = experimental_setup_list,
    value = experimental_setup_list[experimental_setup_index],
    style = {'description_width': '150px'},
    layout = widgets.Layout(width=dropdown_width, height=dropdown_height)
)
experimental_setup_dropdown.observe(experimental_setup_dropdown_callback)
display(experimental_setup_dropdown)

<a name="view_experiment_criteria"></a>
### View the Experiment Compliance Criteria

- Click the `Show Experiment Compliance Criteria` button to view the experiment compliance criteria.


In [ ]:
def show_experiment_criteria_output():
    
    if show_exp_criteria_output_button.description == 'Hide Experiment Compliance Criteria':
        
        exp_criteria_output.clear_output()
        with exp_criteria_output:
            
            if os.path.exists(self_exp_criteria_tabulated_csv):
                print("\n%s: \n" %self_exp_criteria_csv)
                f = open(self_exp_criteria_tabulated_csv,'r')
                for line in f:
                    print(line.rstrip())
                f.close()
            else:
                log_error (log_output, '%s does not exist ' %self_exp_criteria_tabulated_csv + '. Please contact us.')
        
def show_experiment_criteria_output_on_click(change):
    
    if os.path.exists(self_exp_criteria_tabulated_csv):
            
        if show_exp_criteria_output_button.description == 'Show Experiment Compliance Criteria':
            
            show_exp_criteria_output_button.description = 'Hide Experiment Compliance Criteria'
            show_experiment_criteria_output()
            
        else:
        
            show_exp_criteria_output_button.description = 'Show Experiment Compliance Criteria'
            exp_criteria_output.clear_output()
    else:
        log_error (log_output, '%s does not exist ' %self_exp_criteria_tabulated_csv + '. Please contact us.')

show_exp_criteria_output_button = widgets.Button(description="Show Experiment Compliance Criteria", disabled=False,\
    layout=widgets.Layout(width=button_width2, height=button_height2),\
    style= {'button_color':'lightgreen','font_weight':'bold'})

show_exp_criteria_output_button.add_class("buttontextclass")
show_exp_criteria_output_button.on_click(show_experiment_criteria_output_on_click)
display (show_exp_criteria_output_button)

In [ ]:
exp_criteria_output = widgets.Output(layout={'border': widget_output_border_style})
display (exp_criteria_output)

In [ ]:
# Use Hubzero tool session number instead of requests session:
#session = os.environ['USER'] + '-' + str(os.environ['SESSION'])
source_directory = os.path.join(self_workingdir, 'test')
#print ('source_directory: ', source_directory)
upload_directory = os.path.join(source_directory, 'tmpdir')
#print ('upload_directory: ', upload_directory)
exp_directory = ''
compliance_checker_log = os.path.join(source_directory, 'compliance_checker_log.txt')
#print ('compliance_checker_log: ', compliance_checker_log)



<a name="upload_file"></a>
### Upload Your netCDF Formatted Simulation Experiment File

- Upload your netCDF formatted simulation experiment file by clicking the Upload File button.<br />

- After your simulation experiment file is uploaded, the compliance checker python script is automatically executed to check the compliance of the uploaded file.<br/>


In [ ]:
# Called when the list of filenames is received by the FileUpload Widget
def filenames_received_cb(w, exp_ext, exp_filename, exp_var, exp, upload_status): 
 
    global self_exp_ext
    global self_exp_filename
    global self_exp_var
    global self_exp_uploaded

    self_exp_ext = exp_ext
    self_exp_filename = exp_filename
    self_exp_var = exp_var
    self_exp_uploaded = ''
    
    output_widget.clear_output()
    log_widget.clear_output()
    
    with output_widget:
        
        log_info ('filenames_received_cb...')
        #print (type(w)) #<class 'upload.FileUpload'>
        #print (help(w))
        log_info ('exp_ext: ' + self_exp_ext)
        log_info ('exp_filename: ' + exp_filename)
        log_info ('exp_var: ' + self_exp_var)
        
        # Clean up the source directory from a previous run.
        # upload.py will mkdir_p the upload_directory
        if os.path.exists(source_directory):
            shutil.rmtree(source_directory)
        output_widget.clear_output()
        log_widget.clear_output()

        #print (type(upload_status))
        #print ((len(upload_status)))
        #print (upload_status)
        if upload_status[0] == upload_status_check_success:
            
            self_exp_uploaded = exp
            log_info ('exp: ' + exp)
            
        else:
            
            # data_received_cb is not called in this case
            for i in range(len(upload_status)):
                if upload_status[i] == upload_status_ext_check_failed:
                    message = '%s is not a valid netCDF file extension.' %self_exp_ext
                    log_error (output_widget, '%s' %message)
                    message = 'Please upload a netCDF (.nc) file.'
                    log_status (output_widget, '%s' %message)
                elif upload_status[i] == upload_status_exp_check_failed:
                    message = '%s is not a valid ismip6 simulation experiment file for the %s experiment criteria. ' %(self_exp_filename, experimental_setup_list[experimental_setup_index])
                    log_error (output_widget, '%s' %message)
                    message = 'See sections Select the Experiment Compliance Criteria and View the Experiment Compliance Criteria for more information.'
                    log_status (output_widget, '%s' %message)
                elif upload_status[i] == upload_status_var_check_failed:
                    message = '%s is not a ismip6 mandatory variable.' %self_exp_var 
                    log_error (output_widget, '%s' %message)
                    message = 'See section View the Variable Compliance Criteria for more information.'
                    log_status (output_widget, '%s' %message)


In [ ]:
# Called when the file is uploaded by the FileUpload widget
def data_received_cb(w, elasped_time_min): 
    
    global exp_directory
    #global upload
    
    exp_directory = ''
    
    try:
        
        output_widget.clear_output()
        log_widget.clear_output()

        with output_widget:

            log_info ('data_received_cb...')
            #print (type(w)) #<class 'upload.FileUpload'>
            #print (help(w))
            log_info ('elapsed time = ' + str(elasped_time_min) + ' [min]')

            if os.path.exists(upload_directory):
                
                log_info ('exp_filename: ' + self_exp_filename)
                log_info ('exp_uploaded: ' + self_exp_uploaded)

                # Rename upload_directory for the compliance checker.
                # On Ghub, only one file from an experiment directory is uploaded so
                # we do not know the grid resolution of the experiment directory name
                exp_directory = os.path.join(source_directory, self_exp_uploaded)
                log_info ('exp_directory: ' + exp_directory)

                shutil.move(upload_directory, exp_directory)

                if os.path.exists(exp_directory):

                    compliance_checker_ghub(self_experimental_criteria_filename, source_directory)

                    if os.path.exists(compliance_checker_log):

                        with log_widget:

                            print('The compliance_checker_log.txt file can be accessed on theghub.org from the %s directory.\n' %source_directory)

                            print("Reference File: %s.\n" %self_exp_filename)
                            print("%s: \n\n" %compliance_checker_log)
                            f = open(compliance_checker_log,'r')
                            for line in f:
                                print(line.rstrip())
                            f.flush()
                            f.close()
                        #output_widget.clear_output()

                    else:
                        message = 'The compliance checker log file %s does not exist. Please submit a ticket.' %compliance_checker_log
                        log_error (output_widget, '%s' %message)

                else:
                    message = 'The exp directory %s does not exist. Please submit a ticket.' %exp_directory
                    log_error (output_widget, '%s' %message)

            else:
                message = 'The upload directory %s does not exist. Please submit a ticket.' %upload_directory
                log_error (output_widget, '%s' %message)

    except Exception as e:
        message = 'Exception: %s. Please submit a ticket.' %str(e)
        log_error (output_widget, '%s' %message)

    w.reset()
 

In [ ]:
# Create the upload widget.
# Wish list - observe when filename is received by the FileUpload widget.
# Note: upload processing 
def create_upload():
    
    upload = FileUpload("",
           desc = "Please select your netCDF (.nc) file to Upload",
           dir = upload_directory,
           filenames_received_cb = filenames_received_cb,
           data_received_cb = data_received_cb,
           mandatory_variables = self_ismip6_mandatory_variables,
           experiments = self_ismip6_experiments,
           maxnum = 1,
           width = "50%",
           maxsize = '300M')
    return upload


In [ ]:
upload_widget = widgets.Output(layout={'border': '1px solid black'})
display (upload_widget)

In [ ]:
upload = create_upload()
display (upload)


In [ ]:
output_widget = widgets.Output(layout={'border': '1px solid black'})
display (output_widget)

<a name="view_output"></a>
### View and Download Compliance Checker Output

- Output from the compliance checker python script is written to the compliance_checker_log.txt file and displayed below.<br />

- Click the `Download File` button to download compliance_checker_log.txt file to your personal computer.<br />

- If compliance_checker_log.txt is not automatically saved to the Download directory on your personal computer, use your web browser to explicitly save it.<br />



In [ ]:
log_widget = widgets.Output(layout={'border': '1px solid black'})
display (log_widget)

In [ ]:
# Download file from Ghub
display(HTML('<h4>Download File: %s</h4>' %os.path.basename(compliance_checker_log)))
#print (os.path.relpath(pdffilepath, os.getcwd()))
downloadTXTButton = Download(os.path.relpath(compliance_checker_log, os.getcwd()),
    label = 'Download File', style='success', icon='fa-arrow-circle-down')
display(downloadTXTButton)

<a name="view_log_output"></a>
### View Log Output

- If an error is encountered while running this tool, the cause of the error is written to the isschecker_log_file.txt file.

- Click the `Show Log Output` button to view isschecker_log_file.txt.

- Click the `Download File` button to download isschecker_log_file.txt to your personal computer.<br />

- If isschecker_log_file.txt is not automatically saved to the Download directory on your personal computer,  use your web browser to explicitly save it.<br />


In [ ]:
def show_log_output(change):
    
    if os.path.exists(self_log_filepath):
            
        if show_log_output_button.description == 'Show Log Output':
        
            show_log_output_button.description = 'Hide Log Output'
        
            with log_output:
            
                if os.path.exists(self_log_filepath):
                    print("%s: \n\n" %self_log_filepath)
                    f = open(self_log_filepath,'r')
                    for line in f:
                        print(line.rstrip())
                    f.close()
                else:
                    log_error (log_output, '%s does not exist ' %self_log_filepath + '. Please contact us.')
        else:
        
            show_log_output_button.description = 'Show Log Output'
            log_output.clear_output()
    else:
        log_error (log_output, '%s does not exist ' %self_log_filepath + '. Please contact us.')

show_log_output_button.add_class("buttontextclass")
show_log_output_button.on_click(show_log_output)
display (show_log_output_button)

In [ ]:
log_output = widgets.Output(layout={'border': widget_output_border_style})
display (log_output)

In [ ]:
# Download file from Ghub
display(HTML('<h4>Download File: %s</h4>' %os.path.basename(self_log_filepath)))
#print (os.path.relpath(pdffilepath, os.getcwd()))
downloadTXTButton = Download(os.path.relpath(self_log_filepath, os.getcwd()),
    label = 'Download File', style='success', icon='fa-arrow-circle-down')
display(downloadTXTButton)

<a name="background"></a>
## Background

[User Guide](#user_guide) | [Top](#top)

- This Jupyter-based tool uses Python 3.
- This tool is deployed on Debian 10 to run in Tool or App mode style. See https://vhub.org/kb/Development/deploy-styles-for-jupyter-tools for more information.
